In [1]:
from sklearn.model_selection import KFold
import pandas as pd
from torch_geometric.loader import DataLoader
import lightning as L
import torch
import numpy as np

import sys
sys.path.append('../')
import FragShapley

In [2]:
# now the actual training part
regression_datasets = ['O43570_Kd', 'P00915_Kd'][:1]
n_cv = 2
n_explain = 5
expected_value = 'empty'

batch_size = 32

random_state = 42
results_folder = 'graph_regression/'

In [3]:
# load and split data
rows_performance = []
df_expl = pd.DataFrame()

for regression_dataset in regression_datasets:
    print(f'On regression dataset: {regression_dataset}')

    # load dataset
    df = pd.read_csv(f'../0_datasets/regression/{regression_dataset}.csv')

    # cross validation
    cv = KFold(n_splits=n_cv,
               shuffle=True,
               random_state=random_state,
               )
    
    for split, (train_index, test_index) in enumerate(cv.split(df)):
        print(f'\tIn split: {split}')

        # split data and get fingerprints
        smiles_train = df.nonstereo_aromatic_smiles.iloc[train_index].to_list()
        smiles_test = df.nonstereo_aromatic_smiles.iloc[test_index].to_list()
        y_train, y_test = df.pPot_mean.iloc[train_index].to_list(), df.pPot_mean.iloc[test_index].to_list()

        # data into dataloader
        featurizer = FragShapley.Featurizer(input_format='smiles',
                                            output_format='graph')
        ds_train = FragShapley.MoleculeDataset(list_of_inputs=smiles_train,
                                               featurizer=featurizer,
                                               y=y_train)
        ds_test = FragShapley.MoleculeDataset(list_of_inputs=smiles_test,
                                              featurizer=featurizer,
                                              y=y_test)
        dl_train = DataLoader(dataset=ds_train,
                              batch_size=batch_size,
                              )
        dl_test = DataLoader(dataset=ds_test,
                             batch_size=batch_size,
                             )
        # train using lightning
        trainer = L.Trainer(accelerator='gpu',
                            max_epochs=25,
                            )
        model = FragShapley.GCNRegressor()
        trainer.fit(model=model,
                    train_dataloaders=dl_train,
                    )
        y_pred = torch.vstack(trainer.predict(model, dl_test)).detach().cpu().numpy().squeeze()
        rows_performance.append({'dataset': regression_dataset,
                                 'split': split,
                                 'train_index': train_index,
                                 'test_index': test_index,
                                 'y_test': y_test,
                                 'y_pred': y_pred,
                                 })
        
        # now the whole explanation part
        smiles_explain = smiles_test[:n_explain]

        frag_explainer = FragShapley.FragmentExplainer(model=model,
                                                       fragmentation_method='BRICS',
                                                       representation='graph',
                                                       trainer=trainer,
                                                       featurizer=featurizer,
                                                       expected_value=expected_value,
                                                       )
        
        ev_frag = frag_explainer.expected_value
        results_dicts, frag_to_atom_ids = frag_explainer.explain(smiles_explain)
        
        results = {'dataset': regression_dataset,
                   'split': split,
                   'smiles': smiles_explain,
                   'y_true': y_test[:n_explain],
                   'y_pred': y_pred[:n_explain],
                   'fragExplainer_result': results_dicts,
                   'fragExplainer_expected_value': [ev_frag for _ in smiles_explain],
                   'frag_to_atom_ids': frag_to_atom_ids,
                   }
        df_expl_inner = pd.DataFrame(results)
        df_expl = pd.concat((df_expl, df_expl_inner))

df_performance = pd.DataFrame(rows_performance)
df_performance.to_pickle(results_folder + 'df_performance.pkl')
df_expl.to_pickle(results_folder + 'df_explanation.pkl')

On regression dataset: O43570_Kd
	In split: 0


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/jproth/miniconda3/envs/x10_fragShap/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type       | Params | Mode 
-----------------------------------

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=25` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jproth/miniconda3/envs/x10_fragShap/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=5` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [7]:
import seaborn as sns
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

df_performance['RMSE'] = df_performance.apply(lambda x: root_mean_squared_error(y_true=x.y_test, y_pred=x.y_pred), axis=1)
df_performance['MAE'] = df_performance.apply(lambda x: mean_absolute_error(y_true=x.y_test, y_pred=x.y_pred), axis=1)
df_performance['R^2'] = df_performance.apply(lambda x: r2_score(y_true=x.y_test, y_pred=x.y_pred), axis=1)

In [8]:
df_performance

,dataset,split,train_index,test_index,y_test,y_pred,RMSE,MAE,R^2
0,O43570_Kd,0,"[1, 2, 3, 4, 5, 6, 9, 10, 11, 15, 16, 19, 21, ...","[0, 7, 8, 12, 13, 14, 17, 18, 20, 22, 23, 25, ...","[7.137272471682025, 9.0, 6.105794740857917, 7....","[7.383804, 7.432261, 7.482069, 7.6158013, 7.25...",1.061475,0.816471,0.013362
1,O43570_Kd,1,"[0, 7, 8, 12, 13, 14, 17, 18, 20, 22, 23, 25, ...","[1, 2, 3, 4, 5, 6, 9, 10, 11, 15, 16, 19, 21, ...","[6.431798275933005, 6.3872161432802645, 9.0969...","[7.006544, 7.2437954, 7.2654905, 7.2628417, 7....",0.979373,0.771015,0.037870


In [9]:
def check_fragment_values(fragment_dict, expected_value, y_pred):
    return np.allclose((sum(fragment_dict.values()) + expected_value), y_pred)

In [10]:
df_expl['check_fragExplainer_output'] = df_expl.apply(lambda x: check_fragment_values(x.fragExplainer_result, x.fragExplainer_expected_value, x.y_pred), axis=1)
df_expl['check_fragExplainer_output'].value_counts()

check_fragExplainer_output
True    10
Name: count, dtype: int64

In [12]:
df_expl

,dataset,split,smiles,y_true,y_pred,fragExplainer_result,fragExplainer_expected_value,frag_to_atom_ids,check_fragExplainer_output
0,O43570_Kd,0,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,7.137272,7.383804,"{0: 2.7278166611989336, 1: 1.568530241648356, ...",0.288657,"{0: [0, 1, 2, 3, 4, 25, 26], 1: [5, 6, 7, 14, ...",True
1,O43570_Kd,0,C#CCNC(=O)c1ccc(S(N)(=O)=O)cc1,9.000000,7.432261,"{0: 1.3405925432840984, 1: 1.7751912673314412,...",0.288657,"{0: [0, 1, 2], 1: [3], 2: [4, 5], 3: [6, 7, 8,...",True
2,O43570_Kd,0,C#CCNc1ccc(S(N)(=O)=O)cc1,6.105795,7.482069,"{0: 1.8665699164072673, 1: 2.381576379140218, ...",0.288657,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 7, 8, 9, 1...",True
3,O43570_Kd,0,C#CCOP(=O)(Oc1ccc(S(N)(=O)=O)cc1)c1ccccc1,7.547826,7.615801,"{0: 1.955398877461751, 1: 2.681854168574015, 2...",0.288657,"{0: [0, 1, 2], 1: [3, 4, 5, 6, 17, 18, 19, 20,...",True
4,O43570_Kd,0,C#CCOc1c(C=NCc2ccc(S(N)(=O)=O)cc2)cccc1OC,6.180917,7.258069,"{0: 1.076482892036438, 1: 1.5982933203379315, ...",0.288657,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 7, 8, 19, ...",True
0,O43570_Kd,1,C#CCCCOc1ccc2ccc(=O)oc2c1,6.431798,7.006544,"{0: 2.037529150644938, 1: 2.390374898910522, 2...",0.260362,"{0: [0, 1, 2, 3, 4], 1: [5], 2: [6, 7, 8, 9, 1...",True
1,O43570_Kd,1,C#CCCCOc1ccc2ccc(=S)oc2c1,6.387216,7.243795,"{0: 1.9851262569427488, 1: 2.38033135732015, 2...",0.260362,"{0: [0, 1, 2, 3, 4], 1: [5], 2: [6, 7, 8, 9, 1...",True
2,O43570_Kd,1,C#CCN(CC#C)CCCCOc1ccc(S(N)(=O)=O)cc1,9.096910,7.265491,"{0: 0.8471566677093505, 1: 1.182497906684875, ...",0.260362,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6], 3: [7, 8,...",True
3,O43570_Kd,1,C#CCN(CC#C)CCc1ccc(S(N)(=O)=O)cc1,8.040959,7.262842,"{0: 1.0540201107660931, 1: 1.4169544219970702,...",0.260362,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6], 3: [7, 8]...",True
4,O43570_Kd,1,C#CCN(CC#C)c1ccc(S(N)(=O)=O)cc1,8.309804,7.253750,"{0: 1.392299969991048, 1: 1.7081054846445718, ...",0.260362,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6], 3: [7, 8,...",True
